In [ ]:
import gc
import math
import os
import random
import re
import time
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score, matthews_corrcoef, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from imblearn.over_sampling import ADASYN, SMOTE
from imblearn.under_sampling import EditedNearestNeighbours
SEEDS = [11, 22, 33, 44, 55]
PARTITION_SEED = 2026
SAVE_TEST_SETS = True
LABEL_COLUMN = 'label'
BINARY_COLUMN = 'binary_label'
SOURCE_COLUMN = 'source'
ORIGINAL_LABEL_COLUMN = 'original_label'
MAX_TRAIN_ROWS = None
MAX_TEST_ROWS = None
KAMAL_MAX_TRAIN_ROWS = 175368
KAMAL_USE_LOF = True
KAMAL_USE_ADASYN_SMOTE_ENN = True
USE_EARLY_STOPPING = True
EPOCHS = {'kilichev_1dcnn': 99, 'buhl_dcnn': 30, 'kamal_transformer_dnn': 500, 'elmasry_dpso_dnn': 15}
BATCH_SIZES = {'kilichev_1dcnn': 64, 'buhl_dcnn': 128, 'kamal_transformer_dnn': 128, 'elmasry_dpso_dnn': 450}
MODEL_KEYS = ['kilichev_1dcnn', 'buhl_dcnn', 'kamal_transformer_dnn', 'zkr_random_forest', 'elmasry_dpso_dnn']
RESULTS_DIR = Path('/content/drive/MyDrive/Array_submission/Results/CIC_IDS2017/Experiment4_model_specific_test_ratios')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
for gpu in tf.config.list_physical_devices('GPU'):
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass
print('TensorFlow:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))
print('Results directory:', RESULTS_DIR)


TensorFlow: 2.21.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Results directory: /content/drive/MyDrive/Array_submission/Results/CIC_IDS2017/Experiment4_model_specific_test_ratios


In [3]:
def set_all_seeds(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

def binary_y(frame):
    return frame[BINARY_COLUMN].to_numpy(dtype=np.int8)

def optional_sample(frame, maximum_rows, seed):
    if maximum_rows is None or len(frame) <= maximum_rows:
        return frame.reset_index(drop=True).copy()
    sampled, _ = train_test_split(frame, train_size=maximum_rows, random_state=seed, shuffle=True, stratify=frame[BINARY_COLUMN])
    return sampled.reset_index(drop=True)

def make_callbacks(include_reduce_lr=True):
    if not USE_EARLY_STOPPING:
        return []
    callbacks = [tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, min_delta=0.0001, restore_best_weights=True, verbose=1)]
    if include_reduce_lr:
        callbacks.append(tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-07, verbose=1))
    return callbacks


In [4]:
CIC2017_FEATURE_COLUMNS = ['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE Flag Count', 'ECE Flag Count', 'Down/Up Ratio', 'Average Packet Size', 'Avg Fwd Segment Size', 'Avg Bwd Segment Size', 'Fwd Header Length.1', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate', 'Subflow Fwd Packets', 'Subflow Fwd Bytes', 'Subflow Bwd Packets', 'Subflow Bwd Bytes', 'Init_Win_bytes_forward', 'Init_Win_bytes_backward', 'act_data_pkt_fwd', 'min_seg_size_forward', 'Active Mean', 'Active Std', 'Active Max', 'Active Min', 'Idle Mean', 'Idle Std', 'Idle Max', 'Idle Min']
CICFLOWMETER_TO_CIC2017 = {'Dst Port': 'Destination Port', 'Total Fwd Packet': 'Total Fwd Packets', 'Total Bwd packets': 'Total Backward Packets', 'Total Length of Fwd Packet': 'Total Length of Fwd Packets', 'Total Length of Bwd Packet': 'Total Length of Bwd Packets', 'Packet Length Min': 'Min Packet Length', 'Packet Length Max': 'Max Packet Length', 'CWR Flag Count': 'CWE Flag Count', 'Fwd Segment Size Avg': 'Avg Fwd Segment Size', 'Bwd Segment Size Avg': 'Avg Bwd Segment Size', 'Fwd Bytes/Bulk Avg': 'Fwd Avg Bytes/Bulk', 'Fwd Packet/Bulk Avg': 'Fwd Avg Packets/Bulk', 'Fwd Bulk Rate Avg': 'Fwd Avg Bulk Rate', 'Bwd Bytes/Bulk Avg': 'Bwd Avg Bytes/Bulk', 'Bwd Packet/Bulk Avg': 'Bwd Avg Packets/Bulk', 'Bwd Bulk Rate Avg': 'Bwd Avg Bulk Rate', 'FWD Init Win Bytes': 'Init_Win_bytes_forward', 'Bwd Init Win Bytes': 'Init_Win_bytes_backward', 'Fwd Act Data Pkts': 'act_data_pkt_fwd', 'Fwd Seg Size Min': 'min_seg_size_forward'}
UNSW_TO_CIC2017 = {'dst_port': 'Destination Port', 'flow_duration': 'Flow Duration', 'tot_fwd_pkts': 'Total Fwd Packets', 'tot_bwd_pkts': 'Total Backward Packets', 'totlen_fwd_pkts': 'Total Length of Fwd Packets', 'totlen_bwd_pkts': 'Total Length of Bwd Packets', 'fwd_pkt_len_max': 'Fwd Packet Length Max', 'fwd_pkt_len_min': 'Fwd Packet Length Min', 'fwd_pkt_len_mean': 'Fwd Packet Length Mean', 'fwd_pkt_len_std': 'Fwd Packet Length Std', 'bwd_pkt_len_max': 'Bwd Packet Length Max', 'bwd_pkt_len_min': 'Bwd Packet Length Min', 'bwd_pkt_len_mean': 'Bwd Packet Length Mean', 'bwd_pkt_len_std': 'Bwd Packet Length Std', 'flow_byts_s': 'Flow Bytes/s', 'flow_pkts_s': 'Flow Packets/s', 'flow_iat_mean': 'Flow IAT Mean', 'flow_iat_std': 'Flow IAT Std', 'flow_iat_max': 'Flow IAT Max', 'flow_iat_min': 'Flow IAT Min', 'fwd_iat_tot': 'Fwd IAT Total', 'fwd_iat_mean': 'Fwd IAT Mean', 'fwd_iat_std': 'Fwd IAT Std', 'fwd_iat_max': 'Fwd IAT Max', 'fwd_iat_min': 'Fwd IAT Min', 'bwd_iat_tot': 'Bwd IAT Total', 'bwd_iat_mean': 'Bwd IAT Mean', 'bwd_iat_std': 'Bwd IAT Std', 'bwd_iat_max': 'Bwd IAT Max', 'bwd_iat_min': 'Bwd IAT Min', 'fwd_psh_flags': 'Fwd PSH Flags', 'bwd_psh_flags': 'Bwd PSH Flags', 'fwd_urg_flags': 'Fwd URG Flags', 'bwd_urg_flags': 'Bwd URG Flags', 'fwd_header_len': 'Fwd Header Length', 'bwd_header_len': 'Bwd Header Length', 'fwd_pkts_s': 'Fwd Packets/s', 'bwd_pkts_s': 'Bwd Packets/s', 'pkt_len_min': 'Min Packet Length', 'pkt_len_max': 'Max Packet Length', 'pkt_len_mean': 'Packet Length Mean', 'pkt_len_std': 'Packet Length Std', 'pkt_len_var': 'Packet Length Variance', 'fin_flag_cnt': 'FIN Flag Count', 'syn_flag_cnt': 'SYN Flag Count', 'rst_flag_cnt': 'RST Flag Count', 'psh_flag_cnt': 'PSH Flag Count', 'ack_flag_cnt': 'ACK Flag Count', 'urg_flag_cnt': 'URG Flag Count', 'cwr_flag_count': 'CWE Flag Count', 'ece_flag_cnt': 'ECE Flag Count', 'down_up_ratio': 'Down/Up Ratio', 'pkt_size_avg': 'Average Packet Size', 'fwd_seg_size_avg': 'Avg Fwd Segment Size', 'bwd_seg_size_avg': 'Avg Bwd Segment Size', 'fwd_byts_b_avg': 'Fwd Avg Bytes/Bulk', 'fwd_pkts_b_avg': 'Fwd Avg Packets/Bulk', 'fwd_blk_rate_avg': 'Fwd Avg Bulk Rate', 'bwd_byts_b_avg': 'Bwd Avg Bytes/Bulk', 'bwd_pkts_b_avg': 'Bwd Avg Packets/Bulk', 'bwd_blk_rate_avg': 'Bwd Avg Bulk Rate', 'subflow_fwd_pkts': 'Subflow Fwd Packets', 'subflow_fwd_byts': 'Subflow Fwd Bytes', 'subflow_bwd_pkts': 'Subflow Bwd Packets', 'subflow_bwd_byts': 'Subflow Bwd Bytes', 'init_fwd_win_byts': 'Init_Win_bytes_forward', 'init_bwd_win_byts': 'Init_Win_bytes_backward', 'fwd_act_data_pkts': 'act_data_pkt_fwd', 'fwd_seg_size_min': 'min_seg_size_forward', 'active_mean': 'Active Mean', 'active_std': 'Active Std', 'active_max': 'Active Max', 'active_min': 'Active Min', 'idle_mean': 'Idle Mean', 'idle_std': 'Idle Std', 'idle_max': 'Idle Max', 'idle_min': 'Idle Min'}


In [6]:
MODEL_ORIGINAL_TEST_COUNTS = {'kilichev_1dcnn': {'normal': 419252, 'attack': 84908, 'evidence': 'Confusion-matrix row totals: [415925 + 3327 normal, 3956 + 80952 attack]'}, 'buhl_dcnn': {'normal': 567830, 'attack': 139139, 'evidence': 'Confusion-matrix row totals: [566949 + 881 normal, 787 + 138352 attack]'}, 'kamal_transformer_dnn': {'normal': 14035, 'attack': 10955, 'evidence': 'Reported test class counts: 14035 normal and 10955 attack'}, 'zkr_random_forest': {'normal': 454265, 'attack': 111311, 'evidence': 'Confusion-matrix row totals: [452407 + 1858 normal, 23899 + 87412 attack]'}, 'elmasry_dpso_dnn': {'normal': 18750, 'attack': 131250, 'evidence': 'Explicit test sampling: 18750 normal and 131250 attack'}}
MODEL_TEST_NORMAL_RATIO = {model_key: counts['normal'] / (counts['normal'] + counts['attack']) for model_key, counts in MODEL_ORIGINAL_TEST_COUNTS.items()}
MODEL_TEST_ATTACK_RATIO = {model_key: 1.0 - normal_ratio for model_key, normal_ratio in MODEL_TEST_NORMAL_RATIO.items()}
model_ratio_reference = pd.DataFrame([{'model': model_key, 'original_test_normal': counts['normal'], 'original_test_attack': counts['attack'], 'original_test_total': counts['normal'] + counts['attack'], 'target_normal_ratio': MODEL_TEST_NORMAL_RATIO[model_key], 'target_normal_pct': 100 * MODEL_TEST_NORMAL_RATIO[model_key], 'target_attack_ratio': MODEL_TEST_ATTACK_RATIO[model_key], 'target_attack_pct': 100 * MODEL_TEST_ATTACK_RATIO[model_key], 'evidence': counts['evidence']} for model_key, counts in MODEL_ORIGINAL_TEST_COUNTS.items()])

In [ ]:
PARTITION_DIR = Path('/content/drive/MyDrive/Array_submission/Prepared_partitions/splits') / 'cicflow_cic17'
mixed_train_reference = pd.read_csv(PARTITION_DIR / 'balanced_train.csv', low_memory=False)
eligible_test_reference = pd.read_csv(PARTITION_DIR / 'test_all_sources.csv', low_memory=False)
required_columns = set(CIC2017_FEATURE_COLUMNS + [SOURCE_COLUMN, LABEL_COLUMN, BINARY_COLUMN, ORIGINAL_LABEL_COLUMN])
for frame_name, frame in [('balanced_train', mixed_train_reference), ('test_all_sources', eligible_test_reference)]:
    missing = sorted(required_columns - set(frame.columns))
    if missing:
        raise KeyError(f'{frame_name} is missing columns: {missing}')
source_class_counts = mixed_train_reference.groupby([SOURCE_COLUMN, BINARY_COLUMN]).size()
BALANCE_TARGET_PER_CLASS_SOURCE = int(source_class_counts.min())
if not source_class_counts.eq(BALANCE_TARGET_PER_CLASS_SOURCE).all():
    raise AssertionError(f'Loaded training partition is not source/class balanced: {source_class_counts.to_dict()}')
mixed_test_sets = {source: part.reset_index(drop=True) for source, part in eligible_test_reference.groupby(SOURCE_COLUMN, sort=True)}
partition_audit = pd.read_csv(PARTITION_DIR / 'partition_manifest.csv')
training_distribution_export = pd.read_csv(PARTITION_DIR / 'balanced_training_distribution.csv')
split_manifest = pd.DataFrame([{'dataset': 'balanced_training_partition', 'rows': len(mixed_train_reference)}, {'dataset': 'test_partition_before_ratio_matching', 'rows': len(eligible_test_reference)}])
split_manifest.to_csv(RESULTS_DIR / 'mixed_source_split_manifest.csv', index=False)
partition_audit.to_csv(RESULTS_DIR / 'partition_audit.csv', index=False)
training_distribution_export.to_csv(RESULTS_DIR / 'balanced_training_distribution.csv', index=False)
eligible_test_reference.to_csv(RESULTS_DIR / 'test_all_sources_before_ratio_matching.csv', index=False)
print('Loaded balanced training set:', mixed_train_reference.shape)
print('Loaded test partition:', eligible_test_reference.shape)


Loaded balanced training set: (65120, 82)
Loaded test partition: (27912, 82)


In [ ]:
def proportional_group_sample(frame, n_rows, group_column, seed):
    """Sample n_rows while retaining group proportions by largest remainder."""
    if n_rows < 0:
        raise ValueError('n_rows must be non-negative.')
    if n_rows == 0:
        return frame.iloc[0:0].copy()
    if n_rows >= len(frame):
        return frame.sample(frac=1, random_state=seed).reset_index(drop=True)
    working = frame.copy()
    group_values = working[group_column].astype('string').fillna('<missing>')
    counts = group_values.value_counts(dropna=False, sort=False)
    exact = counts / counts.sum() * n_rows
    allocation = np.floor(exact).astype(int)
    allocation = allocation.clip(upper=counts)
    remaining = int(n_rows - allocation.sum())
    while remaining > 0:
        candidates = [group for group in counts.index if allocation[group] < counts[group]]
        if not candidates:
            break
        candidates = sorted(candidates, key=lambda group: (exact[group] - allocation[group], counts[group] - allocation[group], str(group)), reverse=True)
        for group in candidates:
            if remaining == 0:
                break
            if allocation[group] < counts[group]:
                allocation[group] += 1
                remaining -= 1
    sampled_parts = []
    for offset, group in enumerate(counts.index):
        n_group = int(allocation[group])
        if n_group == 0:
            continue
        group_frame = working[group_values.eq(group)]
        sampled_parts.append(group_frame.sample(n=n_group, replace=False, random_state=seed + offset + 1))
    sampled = pd.concat(sampled_parts, ignore_index=True)
    return sampled.sample(frac=1, random_state=seed).reset_index(drop=True)

def match_model_test_ratio(frame, model_key, seed):
    """Downsample a source test set to the model's original binary ratio."""
    target_normal_ratio = MODEL_TEST_NORMAL_RATIO[model_key]
    target_attack_ratio = MODEL_TEST_ATTACK_RATIO[model_key]
    normal_frame = frame[frame[BINARY_COLUMN].eq(0)].copy()
    attack_frame = frame[frame[BINARY_COLUMN].eq(1)].copy()
    available_normal = len(normal_frame)
    available_attack = len(attack_frame)
    if available_normal < 1 or available_attack < 1:
        raise ValueError(f'Ratio matching requires both normal and attack rows. model={model_key}; normal={available_normal}; attack={available_attack}')
    maximum_total = int(math.floor(min(available_normal / target_normal_ratio, available_attack / target_attack_ratio)))
    if maximum_total < 2:
        raise ValueError('Insufficient rows for a two-class matched test set.')
    while maximum_total >= 2:
        selected_normal = int(round(maximum_total * target_normal_ratio))
        selected_attack = maximum_total - selected_normal
        if selected_normal >= 1 and selected_attack >= 1 and (selected_normal <= available_normal) and (selected_attack <= available_attack):
            break
        maximum_total -= 1
    if maximum_total < 2:
        raise ValueError('Could not construct a valid matched test set.')
    sampled_normal = normal_frame.sample(n=selected_normal, replace=False, random_state=seed)
    sampled_attack = proportional_group_sample(attack_frame, n_rows=selected_attack, group_column=LABEL_COLUMN, seed=seed + 10000)
    matched = pd.concat([sampled_normal, sampled_attack], ignore_index=True).sample(frac=1, random_state=seed + 20000).reset_index(drop=True)
    actual_normal = int(matched[BINARY_COLUMN].eq(0).sum())
    actual_attack = int(matched[BINARY_COLUMN].eq(1).sum())
    audit = {'available_rows': int(len(frame)), 'available_normal': int(available_normal), 'available_attack': int(available_attack), 'selected_rows': int(len(matched)), 'selected_normal': actual_normal, 'selected_attack': actual_attack, 'target_normal_ratio': target_normal_ratio, 'target_attack_ratio': target_attack_ratio, 'target_normal_pct': 100 * target_normal_ratio, 'target_attack_pct': 100 * target_attack_ratio, 'actual_normal_ratio': actual_normal / len(matched), 'actual_attack_ratio': actual_attack / len(matched), 'actual_normal_pct': 100 * actual_normal / len(matched), 'actual_attack_pct': 100 * actual_attack / len(matched)}
    return (matched, audit)
model_matched_test_sets = {}
model_test_ratio_audits = {}
model_test_ratio_manifest_rows = []
for model_offset, model_key in enumerate(MODEL_KEYS):
    model_matched_test_sets[model_key] = {}
    for source_offset, (test_source, raw_test_frame) in enumerate(mixed_test_sets.items()):
        matched, audit = match_model_test_ratio(raw_test_frame, model_key=model_key, seed=PARTITION_SEED + 1000 * model_offset + source_offset)
        model_matched_test_sets[model_key][test_source] = matched
        model_test_ratio_audits[model_key, test_source] = audit
        model_test_ratio_manifest_rows.append({'model': model_key, 'test_source': test_source, **audit})
        if SAVE_TEST_SETS:
            matched.to_csv(RESULTS_DIR / f'test_{model_key}_{test_source}.csv', index=False)
    if SAVE_TEST_SETS:
        pd.concat(model_matched_test_sets[model_key].values(), ignore_index=True).to_csv(RESULTS_DIR / f'test_{model_key}_all_sources.csv', index=False)
model_test_ratio_manifest = pd.DataFrame(model_test_ratio_manifest_rows)
model_test_ratio_manifest.to_csv(RESULTS_DIR / 'model_test_ratio_manifest.csv', index=False)

In [9]:
def calculate_binary_metrics(y_true, y_pred, y_score=None):
    y_true = np.asarray(y_true, dtype=np.int8)
    y_pred = np.asarray(y_pred, dtype=np.int8)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if tn + fp > 0 else np.nan
    metrics = {'n_test': int(len(y_true)), 'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp), 'accuracy': accuracy_score(y_true, y_pred), 'balanced_accuracy': balanced_accuracy_score(y_true, y_pred), 'precision': precision_score(y_true, y_pred, pos_label=1, zero_division=0), 'recall': recall_score(y_true, y_pred, pos_label=1, zero_division=0), 'specificity': specificity, 'f1': f1_score(y_true, y_pred, pos_label=1, zero_division=0), 'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0), 'mcc': matthews_corrcoef(y_true, y_pred), 'false_positive_rate': fp / (fp + tn) if fp + tn > 0 else np.nan, 'roc_auc': np.nan}
    if y_score is not None and len(np.unique(y_true)) == 2:
        metrics['roc_auc'] = roc_auc_score(y_true, np.asarray(y_score))
    return metrics

def category_recall_table(frame, y_true, y_pred):
    categories = frame[LABEL_COLUMN].astype(str).to_numpy()
    rows = []
    for category in sorted(pd.unique(categories)):
        mask = categories == category
        category_true = y_true[mask]
        category_prediction = y_pred[mask]
        rows.append({'test_category': category, 'support': int(mask.sum()), 'attack_recall': recall_score(category_true, category_prediction, pos_label=1, zero_division=0) if np.any(category_true == 1) else np.nan, 'benign_specificity': recall_score(category_true, category_prediction, pos_label=0, zero_division=0) if np.any(category_true == 0) else np.nan})
    return pd.DataFrame(rows)

def subtype_recall_table(frame, y_true, y_pred):
    subtypes = frame[ORIGINAL_LABEL_COLUMN].astype(str).str.strip().str.rstrip('.').to_numpy()
    rows = []
    for subtype in sorted(pd.unique(subtypes)):
        mask = subtypes == subtype
        subtype_true = y_true[mask]
        subtype_prediction = y_pred[mask]
        rows.append({'test_subtype': subtype, 'support': int(mask.sum()), 'attack_recall': recall_score(subtype_true, subtype_prediction, pos_label=1, zero_division=0) if np.any(subtype_true == 1) else np.nan, 'benign_specificity': recall_score(subtype_true, subtype_prediction, pos_label=0, zero_division=0) if np.any(subtype_true == 0) else np.nan})
    return pd.DataFrame(rows)


In [10]:
def numeric_matrix(frame, feature_columns):
    matrix = frame.reindex(columns=feature_columns).copy()
    for column in feature_columns:
        matrix[column] = pd.to_numeric(matrix[column], errors='coerce')
    return matrix.replace([np.inf, -np.inf], np.nan)

def fit_numeric_preprocessor(train_frame, feature_columns, scaler_kind, drop_constant=True):
    matrix = numeric_matrix(train_frame, feature_columns)
    medians = matrix.median(axis=0).fillna(0.0)
    matrix = matrix.fillna(medians)
    active_features = list(feature_columns)
    if drop_constant:
        active_features = [column for column in feature_columns if matrix[column].nunique(dropna=False) > 1]
    if not active_features:
        raise ValueError('No non-constant training features remain.')
    matrix = matrix[active_features]
    if scaler_kind == 'standard':
        scaler = StandardScaler()
    elif scaler_kind == 'minmax':
        scaler = MinMaxScaler(feature_range=(0, 1))
    elif scaler_kind == 'none':
        scaler = None
    else:
        raise ValueError(f'Unknown scaler kind: {scaler_kind}')
    transformed = scaler.fit_transform(matrix) if scaler is not None else matrix.to_numpy()
    return ({'feature_columns': active_features, 'medians': medians, 'scaler': scaler, 'scaler_kind': scaler_kind}, np.asarray(transformed, dtype=np.float32))

def transform_numeric_preprocessor(fitted, frame):
    matrix = numeric_matrix(frame, fitted['feature_columns'])
    medians = fitted['medians'].reindex(fitted['feature_columns']).fillna(0.0)
    matrix = matrix.fillna(medians)
    transformed = fitted['scaler'].transform(matrix) if fitted['scaler'] is not None else matrix.to_numpy()
    return np.asarray(transformed, dtype=np.float32)

def prepare_test(frame, seed):
    return optional_sample(frame, MAX_TEST_ROWS, seed)


In [11]:
KILICHEV_DROPPED_FEATURES = {'Bwd PSH Flags', 'Bwd URG Flags', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate'}

def build_kilichev_1dcnn(n_features):
    model = tf.keras.Sequential([tf.keras.layers.Input(shape=(n_features, 1)), tf.keras.layers.Conv1D(filters=64, kernel_size=9, activation='relu'), tf.keras.layers.MaxPooling1D(pool_size=5), tf.keras.layers.Conv1D(filters=128, kernel_size=9, activation='relu'), tf.keras.layers.MaxPooling1D(pool_size=2), tf.keras.layers.Dropout(0.113585), tf.keras.layers.Flatten(), tf.keras.layers.Dense(256, activation='relu'), tf.keras.layers.Dropout(0.113585), tf.keras.layers.Dense(256, activation='relu'), tf.keras.layers.Dropout(0.113585), tf.keras.layers.Dense(1, activation='sigmoid')])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001193), loss='binary_crossentropy', metrics=['accuracy'])
    return model

def fit_kilichev_1dcnn(train_frame, seed):
    set_all_seeds(seed)
    tf.keras.backend.clear_session()
    train_frame = optional_sample(train_frame, MAX_TRAIN_ROWS, seed)
    feature_columns = [column for column in CIC2017_FEATURE_COLUMNS if column not in KILICHEV_DROPPED_FEATURES]
    train_part, validation_part = train_test_split(train_frame, test_size=0.125, random_state=seed, shuffle=True, stratify=train_frame[BINARY_COLUMN])
    preprocessor, X_train = fit_numeric_preprocessor(train_part, feature_columns, scaler_kind='standard', drop_constant=False)
    X_validation = transform_numeric_preprocessor(preprocessor, validation_part)
    model = build_kilichev_1dcnn(X_train.shape[1])
    model.fit(X_train[..., np.newaxis], binary_y(train_part), validation_data=(X_validation[..., np.newaxis], binary_y(validation_part)), epochs=EPOCHS['kilichev_1dcnn'], batch_size=BATCH_SIZES['kilichev_1dcnn'], shuffle=True, callbacks=make_callbacks(), verbose=2)
    return {'model': model, 'preprocessor': preprocessor, 'seed': seed}

def predict_kilichev_1dcnn(fitted, test_frame):
    evaluated = prepare_test(test_frame, fitted['seed'])
    X_test = transform_numeric_preprocessor(fitted['preprocessor'], evaluated)
    probability = fitted['model'].predict(X_test[..., np.newaxis], batch_size=BATCH_SIZES['kilichev_1dcnn'], verbose=0).reshape(-1)
    prediction = (probability >= 0.5).astype(np.int8)
    return (binary_y(evaluated), prediction, probability, evaluated)


In [12]:
def build_buhl_dcnn(n_features):
    model = tf.keras.Sequential([tf.keras.layers.Input(shape=(n_features, 1)), tf.keras.layers.Conv1D(256, 3, strides=1, padding='valid', activation='relu'), tf.keras.layers.MaxPooling1D(pool_size=3, strides=1, padding='valid'), tf.keras.layers.Conv1D(128, 3, strides=1, padding='valid', activation='relu'), tf.keras.layers.UpSampling1D(size=3), tf.keras.layers.Conv1D(256, 3, strides=1, padding='valid', activation='relu'), tf.keras.layers.Flatten(), tf.keras.layers.Dense(256, activation='relu'), tf.keras.layers.Dropout(0.1), tf.keras.layers.Dense(128, activation='relu'), tf.keras.layers.Dropout(0.1), tf.keras.layers.Dense(128, activation='relu'), tf.keras.layers.Dropout(0.1), tf.keras.layers.Dense(1, activation='sigmoid')])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001), loss=tf.keras.losses.BinaryCrossentropy(), metrics=['accuracy'])
    return model

def fit_buhl_dcnn(train_frame, seed):
    set_all_seeds(seed)
    tf.keras.backend.clear_session()
    train_frame = optional_sample(train_frame, MAX_TRAIN_ROWS, seed)
    preprocessor, X_train = fit_numeric_preprocessor(train_frame, CIC2017_FEATURE_COLUMNS, scaler_kind='standard', drop_constant=True)
    model = build_buhl_dcnn(X_train.shape[1])
    model.fit(X_train[..., np.newaxis], binary_y(train_frame), validation_split=0.1, epochs=EPOCHS['buhl_dcnn'], batch_size=BATCH_SIZES['buhl_dcnn'], shuffle=True, callbacks=make_callbacks(), verbose=2)
    return {'model': model, 'preprocessor': preprocessor, 'seed': seed}

def predict_buhl_dcnn(fitted, test_frame):
    evaluated = prepare_test(test_frame, fitted['seed'])
    X_test = transform_numeric_preprocessor(fitted['preprocessor'], evaluated)
    probability = fitted['model'].predict(X_test[..., np.newaxis], batch_size=BATCH_SIZES['buhl_dcnn'], verbose=0).reshape(-1)
    prediction = (probability >= 0.5).astype(np.int8)
    return (binary_y(evaluated), prediction, probability, evaluated)


In [13]:
def build_kamal_transformer_dnn(n_features):
    model_input = tf.keras.Input(shape=(n_features, 1), name='transformer_input')
    attention_output = tf.keras.layers.MultiHeadAttention(num_heads=8, key_dim=64, output_shape=1, name='multi_head_attention')(model_input, model_input)
    attention_output = tf.keras.layers.LayerNormalization(epsilon=1e-06)(attention_output)
    attention_residual = tf.keras.layers.Add()([model_input, attention_output])
    feed_forward = tf.keras.layers.Dense(512, activation='relu')(attention_residual)
    feed_forward = tf.keras.layers.Dense(512, activation=None)(feed_forward)
    projected_residual = tf.keras.layers.Dense(512, activation=None)(attention_residual)
    transformer_output = tf.keras.layers.Add()([projected_residual, feed_forward])
    transformer_output = tf.keras.layers.LayerNormalization(epsilon=1e-06)(transformer_output)
    x = tf.keras.layers.Flatten()(transformer_output)
    x = tf.keras.layers.Dense(256, activation='relu')(x)
    x = tf.keras.layers.Dropout(1e-07)(x)
    x = tf.keras.layers.Dense(128, activation='relu')(x)
    x = tf.keras.layers.Dropout(1e-07)(x)
    x = tf.keras.layers.Dense(64, activation='relu')(x)
    output = tf.keras.layers.Dense(1, activation='sigmoid')(x)
    model = tf.keras.Model(model_input, output, name='kamal_transformer_dnn')
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss=tf.keras.losses.BinaryCrossentropy(), metrics=['accuracy'])
    return model

def kamal_clean_and_resample(X_train, y_train, seed):
    X_result = np.asarray(X_train, dtype=np.float32)
    y_result = np.asarray(y_train, dtype=np.int8)
    if KAMAL_USE_LOF and len(X_result) > 20:
        lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05, novelty=False, n_jobs=-1)
        inlier_mask = lof.fit_predict(X_result) == 1
        if len(np.unique(y_result[inlier_mask])) == 2:
            X_result = X_result[inlier_mask]
            y_result = y_result[inlier_mask]
    if not KAMAL_USE_ADASYN_SMOTE_ENN:
        return (X_result, y_result)
    counts = pd.Series(y_result).value_counts()
    if len(counts) == 2 and counts.min() >= 6 and (counts.min() < counts.max()):
        try:
            X_result, y_result = ADASYN(sampling_strategy='auto', random_state=seed, n_neighbors=5).fit_resample(X_result, y_result)
        except ValueError as error:
            warnings.warn(f'ADASYN skipped: {error}')
    counts = pd.Series(y_result).value_counts()
    if len(counts) == 2 and counts.min() >= 6 and (counts.min() < counts.max()):
        minority_label = int(counts.idxmin())
        majority_count = int(counts.max())
        try:
            X_result, y_result = SMOTE(sampling_strategy={minority_label: majority_count}, random_state=seed, k_neighbors=5).fit_resample(X_result, y_result)
        except ValueError as error:
            warnings.warn(f'SMOTE skipped: {error}')
    if len(X_result) > 3:
        try:
            X_result, y_result = EditedNearestNeighbours(sampling_strategy='all', n_neighbors=3, kind_sel='all', n_jobs=-1).fit_resample(X_result, y_result)
        except ValueError as error:
            warnings.warn(f'ENN skipped: {error}')
    return (np.asarray(X_result, dtype=np.float32), np.asarray(y_result, dtype=np.int8))

def fit_kamal_transformer_dnn(train_frame, seed):
    set_all_seeds(seed)
    tf.keras.backend.clear_session()
    train_frame = optional_sample(train_frame, KAMAL_MAX_TRAIN_ROWS, seed)
    preprocessor, X_train = fit_numeric_preprocessor(train_frame, CIC2017_FEATURE_COLUMNS, scaler_kind='minmax', drop_constant=True)
    X_train, y_train = kamal_clean_and_resample(X_train, binary_y(train_frame), seed)
    model = build_kamal_transformer_dnn(X_train.shape[1])
    model.fit(X_train[..., np.newaxis], y_train, validation_split=0.1, epochs=EPOCHS['kamal_transformer_dnn'], batch_size=BATCH_SIZES['kamal_transformer_dnn'], shuffle=True, callbacks=make_callbacks(), verbose=2)
    return {'model': model, 'preprocessor': preprocessor, 'seed': seed}

def predict_kamal_transformer_dnn(fitted, test_frame):
    evaluated = prepare_test(test_frame, fitted['seed'])
    X_test = transform_numeric_preprocessor(fitted['preprocessor'], evaluated)
    probability = fitted['model'].predict(X_test[..., np.newaxis], batch_size=BATCH_SIZES['kamal_transformer_dnn'], verbose=0).reshape(-1)
    prediction = (probability >= 0.5).astype(np.int8)
    return (binary_y(evaluated), prediction, probability, evaluated)


In [14]:
def fit_zkr_random_forest(train_frame, seed):
    set_all_seeds(seed)
    train_frame = optional_sample(train_frame, MAX_TRAIN_ROWS, seed)
    preprocessor, X_train_z = fit_numeric_preprocessor(train_frame, CIC2017_FEATURE_COLUMNS, scaler_kind='standard', drop_constant=True)
    y_train = binary_y(train_frame)
    inlier_mask = (np.abs(X_train_z) <= 3).all(axis=1)
    if inlier_mask.sum() >= 20 and len(np.unique(y_train[inlier_mask])) == 2:
        X_inliers = X_train_z[inlier_mask]
        y_inliers = y_train[inlier_mask]
    else:
        X_inliers = X_train_z
        y_inliers = y_train
    n_components = min(10, X_inliers.shape[1], X_inliers.shape[0] - 1)
    pca = PCA(n_components=n_components, random_state=seed)
    X_pca = pca.fit_transform(X_inliers).astype(np.float32)
    kmeans = KMeans(n_clusters=2, random_state=seed, n_init=10)
    cluster_ids = kmeans.fit_predict(X_pca)
    X_zkr = np.column_stack([X_pca, cluster_ids]).astype(np.float32)
    model = RandomForestClassifier(n_estimators=100, random_state=seed, n_jobs=-1)
    model.fit(X_zkr, y_inliers)
    return {'model': model, 'preprocessor': preprocessor, 'pca': pca, 'kmeans': kmeans, 'seed': seed}

def predict_zkr_random_forest(fitted, test_frame):
    evaluated = prepare_test(test_frame, fitted['seed'])
    X_test_z = transform_numeric_preprocessor(fitted['preprocessor'], evaluated)
    X_test_pca = fitted['pca'].transform(X_test_z).astype(np.float32)
    cluster_ids = fitted['kmeans'].predict(X_test_pca)
    X_test_zkr = np.column_stack([X_test_pca, cluster_ids]).astype(np.float32)
    prediction = fitted['model'].predict(X_test_zkr).astype(np.int8)
    probability = fitted['model'].predict_proba(X_test_zkr)[:, 1]
    return (binary_y(evaluated), prediction, probability, evaluated)


In [15]:
ELMASRY_ONE_BASED_FEATURE_INDICES = [8, 11, 15, 18, 20, 23, 24, 26, 28, 31, 33, 37, 43, 44, 51, 53, 54, 59, 70, 73, 74, 77]
ELMASRY_FEATURE_COLUMNS = [CIC2017_FEATURE_COLUMNS[index - 1] for index in ELMASRY_ONE_BASED_FEATURE_INDICES]

def build_elmasry_dpso_dnn():
    learning_rate_schedule = tf.keras.optimizers.schedules.InverseTimeDecay(initial_learning_rate=0.1, decay_steps=1, decay_rate=0.005, staircase=False)
    model = tf.keras.Sequential([tf.keras.layers.Input(shape=(22,)), tf.keras.layers.Dense(23, activation='tanh', kernel_initializer=tf.keras.initializers.RandomUniform(minval=-0.05, maxval=0.05, seed=0)), tf.keras.layers.Dropout(0.25, seed=0), tf.keras.layers.Dense(15, activation='softsign', kernel_initializer=tf.keras.initializers.RandomUniform(minval=-0.05, maxval=0.05, seed=1)), tf.keras.layers.Dropout(0.25, seed=1), tf.keras.layers.Dense(6, activation='softplus', kernel_initializer=tf.keras.initializers.RandomUniform(minval=-0.05, maxval=0.05, seed=2)), tf.keras.layers.Dropout(0.25, seed=2), tf.keras.layers.Dense(2, activation='softmax', kernel_initializer=tf.keras.initializers.RandomUniform(minval=-0.05, maxval=0.05, seed=3))])
    model.compile(optimizer=tf.keras.optimizers.RMSprop(learning_rate=learning_rate_schedule, momentum=0.3), loss=tf.keras.losses.SparseCategoricalCrossentropy(), metrics=['accuracy'])
    return model

def fit_elmasry_dpso_dnn(train_frame, seed):
    set_all_seeds(seed)
    tf.keras.backend.clear_session()
    train_frame = optional_sample(train_frame, MAX_TRAIN_ROWS, seed)
    preprocessor, X_train = fit_numeric_preprocessor(train_frame, ELMASRY_FEATURE_COLUMNS, scaler_kind='minmax', drop_constant=False)
    model = build_elmasry_dpso_dnn()
    model.fit(X_train, binary_y(train_frame), validation_split=0.1, epochs=EPOCHS['elmasry_dpso_dnn'], batch_size=BATCH_SIZES['elmasry_dpso_dnn'], shuffle=True, callbacks=make_callbacks(include_reduce_lr=False), verbose=2)
    return {'model': model, 'preprocessor': preprocessor, 'seed': seed}

def predict_elmasry_dpso_dnn(fitted, test_frame):
    evaluated = prepare_test(test_frame, fitted['seed'])
    X_test = transform_numeric_preprocessor(fitted['preprocessor'], evaluated)
    probabilities = fitted['model'].predict(X_test, batch_size=BATCH_SIZES['elmasry_dpso_dnn'], verbose=0)
    prediction = np.argmax(probabilities, axis=1).astype(np.int8)
    return (binary_y(evaluated), prediction, probabilities[:, 1], evaluated)


In [16]:
MODEL_REGISTRY = {'kilichev_1dcnn': {'fit': fit_kilichev_1dcnn, 'predict': predict_kilichev_1dcnn}, 'buhl_dcnn': {'fit': fit_buhl_dcnn, 'predict': predict_buhl_dcnn}, 'kamal_transformer_dnn': {'fit': fit_kamal_transformer_dnn, 'predict': predict_kamal_transformer_dnn}, 'zkr_random_forest': {'fit': fit_zkr_random_forest, 'predict': predict_zkr_random_forest}, 'elmasry_dpso_dnn': {'fit': fit_elmasry_dpso_dnn, 'predict': predict_elmasry_dpso_dnn}}
missing_models = [model for model in MODEL_KEYS if model not in MODEL_REGISTRY]
if missing_models:
    raise KeyError(f'Unregistered models: {missing_models}')
print('Registered models:', list(MODEL_REGISTRY))


Registered models: ['kilichev_1dcnn', 'buhl_dcnn', 'kamal_transformer_dnn', 'zkr_random_forest', 'elmasry_dpso_dnn']


In [17]:
def merge_and_save(path, new_rows, key_columns):
    path = Path(path)
    if path.exists():
        existing = pd.read_csv(path)
        combined = pd.concat([existing, new_rows], ignore_index=True)
    else:
        combined = new_rows.copy()
    combined = combined.drop_duplicates(subset=key_columns, keep='last').reset_index(drop=True)
    combined.to_csv(path, index=False)
    return combined


In [18]:
MIXED_EXPERIMENT = 'mixed_source_cic17'
TRAINING_VARIANT = 'mixed_source_balanced_65120'

def evaluate_mixed_source_model(model_key, seed):
    adapter = MODEL_REGISTRY[model_key]
    train_frame = mixed_train_reference
    source_class_quota = BALANCE_TARGET_PER_CLASS_SOURCE
    target_normal_ratio = MODEL_TEST_NORMAL_RATIO[model_key]
    target_attack_ratio = MODEL_TEST_ATTACK_RATIO[model_key]
    print(f'\n{MIXED_EXPERIMENT} | {model_key} | seed={seed}', flush=True)
    print(f'  Target test ratio: {100 * target_normal_ratio:.4f}% normal / {100 * target_attack_ratio:.4f}% attack', flush=True)
    print(f'  Balanced training rows: {len(train_frame):,}; quota={source_class_quota:,}', flush=True)
    fit_started = time.perf_counter()
    fitted = adapter['fit'](train_frame, seed)
    fit_seconds = time.perf_counter() - fit_started
    overall_rows = []
    category_frames = []
    subtype_frames = []
    ratio_rows = []
    pooled_true = []
    pooled_prediction = []
    pooled_score = []
    source_items = list(model_matched_test_sets[model_key].items())
    for source_index, (test_source, matched_test_frame) in enumerate(source_items):
        raw_test_frame = mixed_test_sets[test_source]
        ratio_audit = model_test_ratio_audits[model_key, test_source].copy()
        print(f'  Testing {test_source}: {len(raw_test_frame):,} available -> {len(matched_test_frame):,} matched; normal={ratio_audit['actual_normal_pct']:.4f}%; attack={ratio_audit['actual_attack_pct']:.4f}%', flush=True)
        prediction_started = time.perf_counter()
        y_true, y_pred, y_score, evaluated = adapter['predict'](fitted, matched_test_frame)
        prediction_seconds = time.perf_counter() - prediction_started
        metrics = calculate_binary_metrics(y_true, y_pred, y_score)
        evaluated_normal = int(np.sum(y_true == 0))
        evaluated_attack = int(np.sum(y_true == 1))
        overall_rows.append({'experiment': MIXED_EXPERIMENT, 'training_variant': TRAINING_VARIANT, 'model': model_key, 'seed': seed, 'test_source': test_source, 'source_class_quota': source_class_quota, 'n_train_used': len(train_frame), 'n_test_raw': len(raw_test_frame), 'n_test_ratio_matched': len(matched_test_frame), 'n_test_evaluated': len(y_true), 'target_normal_ratio': target_normal_ratio, 'target_attack_ratio': target_attack_ratio, 'actual_normal_ratio': evaluated_normal / len(y_true), 'actual_attack_ratio': evaluated_attack / len(y_true), 'test_normal_pct': 100 * evaluated_normal / len(y_true), 'test_attack_pct': 100 * evaluated_attack / len(y_true), 'fit_seconds': fit_seconds, 'prediction_seconds': prediction_seconds, **metrics})
        ratio_rows.append({'experiment': MIXED_EXPERIMENT, 'training_variant': TRAINING_VARIANT, 'model': model_key, 'seed': seed, 'test_source': test_source, **ratio_audit})
        category_table = category_recall_table(evaluated, y_true, y_pred)
        category_table['experiment'] = MIXED_EXPERIMENT
        category_table['training_variant'] = TRAINING_VARIANT
        category_table['model'] = model_key
        category_table['seed'] = seed
        category_table['test_source'] = test_source
        category_frames.append(category_table)
        subtype_table = subtype_recall_table(evaluated, y_true, y_pred)
        subtype_table['experiment'] = MIXED_EXPERIMENT
        subtype_table['training_variant'] = TRAINING_VARIANT
        subtype_table['model'] = model_key
        subtype_table['seed'] = seed
        subtype_table['test_source'] = test_source
        subtype_frames.append(subtype_table)
        pooled_true.append(y_true)
        pooled_prediction.append(y_pred)
        pooled_score.append(y_score)
    pooled_true_array = np.concatenate(pooled_true)
    pooled_prediction_array = np.concatenate(pooled_prediction)
    pooled_score_array = np.concatenate(pooled_score)
    pooled_metrics = calculate_binary_metrics(pooled_true_array, pooled_prediction_array, pooled_score_array)
    pooled_normal = int(np.sum(pooled_true_array == 0))
    pooled_attack = int(np.sum(pooled_true_array == 1))
    overall_rows.append({'experiment': MIXED_EXPERIMENT, 'training_variant': TRAINING_VARIANT, 'model': model_key, 'seed': seed, 'test_source': 'POOLED_ALL_ROWS', 'source_class_quota': source_class_quota, 'n_train_used': len(train_frame), 'n_test_raw': int(sum((len(frame) for frame in mixed_test_sets.values()))), 'n_test_ratio_matched': int(sum((len(values) for values in pooled_true))), 'n_test_evaluated': int(len(pooled_true_array)), 'target_normal_ratio': target_normal_ratio, 'target_attack_ratio': target_attack_ratio, 'actual_normal_ratio': pooled_normal / len(pooled_true_array), 'actual_attack_ratio': pooled_attack / len(pooled_true_array), 'test_normal_pct': 100 * pooled_normal / len(pooled_true_array), 'test_attack_pct': 100 * pooled_attack / len(pooled_true_array), 'fit_seconds': fit_seconds, 'prediction_seconds': np.nan, **pooled_metrics})
    del fitted
    gc.collect()
    tf.keras.backend.clear_session()
    return (pd.DataFrame(overall_rows), pd.concat(category_frames, ignore_index=True), pd.concat(subtype_frames, ignore_index=True), pd.DataFrame(ratio_rows))


In [19]:
def run_mixed_source_experiment():
    overall_path = RESULTS_DIR / f'{MIXED_EXPERIMENT}_all_runs_partial.csv'
    category_path = RESULTS_DIR / f'{MIXED_EXPERIMENT}_category_runs_partial.csv'
    subtype_path = RESULTS_DIR / f'{MIXED_EXPERIMENT}_subtype_runs_partial.csv'
    ratio_path = RESULTS_DIR / f'{MIXED_EXPERIMENT}_test_ratio_manifest_partial.csv'
    completed = pd.read_csv(overall_path) if overall_path.exists() else pd.DataFrame()
    overall_accumulated = completed.copy()
    category_accumulated = pd.read_csv(category_path) if category_path.exists() else pd.DataFrame()
    subtype_accumulated = pd.read_csv(subtype_path) if subtype_path.exists() else pd.DataFrame()
    ratio_accumulated = pd.read_csv(ratio_path) if ratio_path.exists() else pd.DataFrame()
    for model_key in MODEL_KEYS:
        for seed in SEEDS:
            if not completed.empty:
                complete_mask = completed['experiment'].eq(MIXED_EXPERIMENT) & completed['model'].eq(model_key) & completed['seed'].eq(seed) & completed['test_source'].eq('POOLED_ALL_ROWS')
                is_complete = bool(complete_mask.any())
            else:
                is_complete = False
            if is_complete:
                print(f'SKIP completed: {model_key}, seed={seed}', flush=True)
                continue
            overall, categories, subtypes, ratio_manifest = evaluate_mixed_source_model(model_key, seed)
            category_accumulated = merge_and_save(category_path, categories, ['experiment', 'training_variant', 'model', 'seed', 'test_source', 'test_category'])
            subtype_accumulated = merge_and_save(subtype_path, subtypes, ['experiment', 'training_variant', 'model', 'seed', 'test_source', 'test_subtype'])
            ratio_accumulated = merge_and_save(ratio_path, ratio_manifest, ['experiment', 'training_variant', 'model', 'seed', 'test_source'])
            overall_accumulated = merge_and_save(overall_path, overall, ['experiment', 'training_variant', 'model', 'seed', 'test_source'])
            completed = overall_accumulated
    return (overall_accumulated, category_accumulated, subtype_accumulated, ratio_accumulated)


In [20]:
mixed_overall, mixed_categories, mixed_subtypes, mixed_ratio_manifest = run_mixed_source_experiment()



mixed_source_cic17 | kilichev_1dcnn | seed=11
  Target test ratio: 83.1585% normal / 16.8415% attack
  Balanced training rows: 65,120; quota=8,140
Epoch 1/99
891/891 - 26s - 29ms/step - accuracy: 0.8889 - loss: 0.2726 - val_accuracy: 0.9055 - val_loss: 0.2181 - learning_rate: 0.0012
Epoch 2/99
891/891 - 3s - 3ms/step - accuracy: 0.9145 - loss: 0.2111 - val_accuracy: 0.9108 - val_loss: 0.2134 - learning_rate: 0.0012
Epoch 3/99
891/891 - 3s - 3ms/step - accuracy: 0.9184 - loss: 0.1992 - val_accuracy: 0.9134 - val_loss: 0.2015 - learning_rate: 0.0012
Epoch 4/99
891/891 - 3s - 3ms/step - accuracy: 0.9202 - loss: 0.1902 - val_accuracy: 0.9147 - val_loss: 0.1844 - learning_rate: 0.0012
Epoch 5/99
891/891 - 2s - 3ms/step - accuracy: 0.9212 - loss: 0.1821 - val_accuracy: 0.9125 - val_loss: 0.1926 - learning_rate: 0.0012
Epoch 6/99
891/891 - 2s - 3ms/step - accuracy: 0.9226 - loss: 0.1738 - val_accuracy: 0.9154 - val_loss: 0.1731 - learning_rate: 0.0012
Epoch 7/99
891/891 - 2s - 3ms/step - acc

/usr/local/lib/python3.13/dist-packages/sklearn/neighbors/_lof.py:322: UserWarning: Duplicate values are leading to incorrect results. Increase the number of neighbors for more accurate results.
  warnings.warn(
/tmp/ipykernel_1706/1824509822.py:38: UserWarning: ADASYN skipped: No samples will be generated with the provided ratio settings.
  warnings.warn(f'ADASYN skipped: {error}')


Epoch 1/500
374/374 - 59s - 157ms/step - accuracy: 0.9011 - loss: 0.4502 - val_accuracy: 0.9667 - val_loss: 0.1139 - learning_rate: 0.0010
Epoch 2/500
374/374 - 2s - 5ms/step - accuracy: 0.9690 - loss: 0.0964 - val_accuracy: 0.9695 - val_loss: 0.0965 - learning_rate: 0.0010
Epoch 3/500
374/374 - 2s - 5ms/step - accuracy: 0.9793 - loss: 0.0697 - val_accuracy: 0.9694 - val_loss: 0.1044 - learning_rate: 0.0010
Epoch 4/500
374/374 - 2s - 5ms/step - accuracy: 0.9832 - loss: 0.0558 - val_accuracy: 0.9709 - val_loss: 0.0822 - learning_rate: 0.0010
Epoch 5/500
374/374 - 2s - 5ms/step - accuracy: 0.9858 - loss: 0.0469 - val_accuracy: 0.9780 - val_loss: 0.0744 - learning_rate: 0.0010
Epoch 6/500
374/374 - 2s - 5ms/step - accuracy: 0.9882 - loss: 0.0406 - val_accuracy: 0.9684 - val_loss: 0.0959 - learning_rate: 0.0010
Epoch 7/500
374/374 - 2s - 5ms/step - accuracy: 0.9881 - loss: 0.0399 - val_accuracy: 0.9737 - val_loss: 0.0841 - learning_rate: 0.0010
Epoch 8/500
374/374 - 2s - 4ms/step - accurac

/usr/local/lib/python3.13/dist-packages/sklearn/neighbors/_lof.py:322: UserWarning: Duplicate values are leading to incorrect results. Increase the number of neighbors for more accurate results.
  warnings.warn(
/tmp/ipykernel_1706/1824509822.py:38: UserWarning: ADASYN skipped: No samples will be generated with the provided ratio settings.
  warnings.warn(f'ADASYN skipped: {error}')


Epoch 1/500
374/374 - 47s - 125ms/step - accuracy: 0.8894 - loss: 0.3861 - val_accuracy: 0.9707 - val_loss: 0.1034 - learning_rate: 0.0010
Epoch 2/500
374/374 - 2s - 5ms/step - accuracy: 0.9148 - loss: 0.3092 - val_accuracy: 0.9693 - val_loss: 0.1069 - learning_rate: 0.0010
Epoch 3/500
374/374 - 2s - 5ms/step - accuracy: 0.9596 - loss: 0.1218 - val_accuracy: 0.9722 - val_loss: 0.0999 - learning_rate: 0.0010
Epoch 4/500
374/374 - 2s - 4ms/step - accuracy: 0.9727 - loss: 0.0882 - val_accuracy: 0.9616 - val_loss: 0.1125 - learning_rate: 0.0010
Epoch 5/500
374/374 - 2s - 5ms/step - accuracy: 0.9782 - loss: 0.0732 - val_accuracy: 0.9782 - val_loss: 0.0694 - learning_rate: 0.0010
Epoch 6/500
374/374 - 2s - 5ms/step - accuracy: 0.9810 - loss: 0.0630 - val_accuracy: 0.9744 - val_loss: 0.0742 - learning_rate: 0.0010
Epoch 7/500
374/374 - 2s - 4ms/step - accuracy: 0.9826 - loss: 0.0575 - val_accuracy: 0.9742 - val_loss: 0.0713 - learning_rate: 0.0010
Epoch 8/500
374/374 - 2s - 5ms/step - accurac

...

Epoch 1/500
374/374 - 42s - 112ms/step - accuracy: 0.9014 - loss: 0.4121 - val_accuracy: 0.9693 - val_loss: 0.1181 - learning_rate: 0.0010
Epoch 2/500
374/374 - 2s - 5ms/step - accuracy: 0.9670 - loss: 0.1002 - val_accuracy: 0.9825 - val_loss: 0.0585 - learning_rate: 0.0010
Epoch 3/500
374/374 - 2s - 5ms/step - accuracy: 0.9792 - loss: 0.0673 - val_accuracy: 0.9739 - val_loss: 0.0762 - learning_rate: 0.0010
Epoch 4/500
374/374 - 2s - 5ms/step - accuracy: 0.9840 - loss: 0.0546 - val_accuracy: 0.9812 - val_loss: 0.0549 - learning_rate: 0.0010
Epoch 5/500
374/374 - 2s - 5ms/step - accuracy: 0.9860 - loss: 0.0487 - val_accuracy: 0.9776 - val_loss: 0.0747 - learning_rate: 0.0010
Epoch 6/500
374/374 - 2s - 5ms/step - accuracy: 0.9859 - loss: 0.0483 - val_accuracy: 0.9675 - val_loss: 0.0851 - learning_rate: 0.0010
Epoch 7/500
374/374 - 2s - 5ms/step - accuracy: 0.9880 - loss: 0.0414 - val_accuracy: 0.9812 - val_loss: 0.0653 - learning_rate: 0.0010
Epoch 8/500
374/374 - 2s - 4ms/step - accurac

/usr/local/lib/python3.13/dist-packages/sklearn/neighbors/_lof.py:322: UserWarning: Duplicate values are leading to incorrect results. Increase the number of neighbors for more accurate results.
  warnings.warn(
/tmp/ipykernel_1706/1824509822.py:38: UserWarning: ADASYN skipped: No samples will be generated with the provided ratio settings.
  warnings.warn(f'ADASYN skipped: {error}')


Epoch 1/500
374/374 - 47s - 127ms/step - accuracy: 0.8677 - loss: 0.4811 - val_accuracy: 0.9635 - val_loss: 0.1295 - learning_rate: 0.0010
Epoch 2/500
374/374 - 2s - 5ms/step - accuracy: 0.9538 - loss: 0.1437 - val_accuracy: 0.9693 - val_loss: 0.0985 - learning_rate: 0.0010
Epoch 3/500
374/374 - 2s - 5ms/step - accuracy: 0.9786 - loss: 0.0723 - val_accuracy: 0.9748 - val_loss: 0.0795 - learning_rate: 0.0010
Epoch 4/500
374/374 - 2s - 5ms/step - accuracy: 0.9817 - loss: 0.0601 - val_accuracy: 0.9757 - val_loss: 0.0836 - learning_rate: 0.0010
Epoch 5/500
374/374 - 2s - 5ms/step - accuracy: 0.9841 - loss: 0.0528 - val_accuracy: 0.9795 - val_loss: 0.0688 - learning_rate: 0.0010
Epoch 6/500
374/374 - 2s - 5ms/step - accuracy: 0.9864 - loss: 0.0461 - val_accuracy: 0.9810 - val_loss: 0.0602 - learning_rate: 0.0010
Epoch 7/500
374/374 - 2s - 5ms/step - accuracy: 0.9868 - loss: 0.0453 - val_accuracy: 0.9797 - val_loss: 0.0620 - learning_rate: 0.0010
Epoch 8/500
374/374 - 2s - 4ms/step - accurac

/usr/local/lib/python3.13/dist-packages/sklearn/neighbors/_lof.py:322: UserWarning: Duplicate values are leading to incorrect results. Increase the number of neighbors for more accurate results.
  warnings.warn(
/tmp/ipykernel_1706/1824509822.py:38: UserWarning: ADASYN skipped: No samples will be generated with the provided ratio settings.
  warnings.warn(f'ADASYN skipped: {error}')


Epoch 1/500
374/374 - 40s - 108ms/step - accuracy: 0.8716 - loss: 0.5705 - val_accuracy: 0.9778 - val_loss: 0.0751 - learning_rate: 0.0010
Epoch 2/500
374/374 - 2s - 5ms/step - accuracy: 0.9669 - loss: 0.1031 - val_accuracy: 0.9712 - val_loss: 0.0866 - learning_rate: 0.0010
Epoch 3/500
374/374 - 2s - 5ms/step - accuracy: 0.9793 - loss: 0.0678 - val_accuracy: 0.9714 - val_loss: 0.0963 - learning_rate: 0.0010
Epoch 4/500
374/374 - 2s - 5ms/step - accuracy: 0.9841 - loss: 0.0549 - val_accuracy: 0.9784 - val_loss: 0.0723 - learning_rate: 0.0010
Epoch 5/500
374/374 - 2s - 5ms/step - accuracy: 0.9860 - loss: 0.0473 - val_accuracy: 0.9846 - val_loss: 0.0523 - learning_rate: 0.0010
Epoch 6/500
374/374 - 2s - 5ms/step - accuracy: 0.9863 - loss: 0.0466 - val_accuracy: 0.9842 - val_loss: 0.0464 - learning_rate: 0.0010
Epoch 7/500
374/374 - 2s - 5ms/step - accuracy: 0.9869 - loss: 0.0439 - val_accuracy: 0.9825 - val_loss: 0.0603 - learning_rate: 0.0010
Epoch 8/500
374/374 - 2s - 5ms/step - accurac

In [21]:
MIXED_METRICS = ['accuracy', 'balanced_accuracy', 'precision', 'recall', 'specificity', 'f1', 'macro_f1', 'mcc', 'false_positive_rate', 'roc_auc']
mixed_per_source_summary = mixed_overall.groupby(['experiment', 'training_variant', 'model', 'test_source'], dropna=False)[MIXED_METRICS].agg(['mean', 'std', 'min', 'max']).reset_index()
mixed_source_macro_by_seed = mixed_overall[mixed_overall['test_source'].ne('POOLED_ALL_ROWS')].groupby(['experiment', 'training_variant', 'model', 'seed'], dropna=False).agg(source_macro_f1=('f1', 'mean'), source_macro_balanced_accuracy=('balanced_accuracy', 'mean'), source_macro_recall=('recall', 'mean'), source_macro_specificity=('specificity', 'mean')).reset_index()
mixed_source_macro_summary = mixed_source_macro_by_seed.groupby(['experiment', 'training_variant', 'model'], dropna=False)[['source_macro_f1', 'source_macro_balanced_accuracy', 'source_macro_recall', 'source_macro_specificity']].agg(['mean', 'std', 'min', 'max']).reset_index()
mixed_category_summary = mixed_categories.groupby(['experiment', 'training_variant', 'model', 'test_source', 'test_category'], dropna=False).agg(support=('support', 'first'), attack_recall_mean=('attack_recall', 'mean'), attack_recall_std=('attack_recall', 'std'), benign_specificity_mean=('benign_specificity', 'mean'), benign_specificity_std=('benign_specificity', 'std')).reset_index()
ratio_summary = mixed_ratio_manifest.groupby(['experiment', 'training_variant', 'model', 'test_source'], dropna=False).agg(target_normal_pct=('target_normal_pct', 'first'), target_attack_pct=('target_attack_pct', 'first'), available_rows=('available_rows', 'first'), selected_rows_mean=('selected_rows', 'mean'), actual_normal_pct_mean=('actual_normal_pct', 'mean'), actual_normal_pct_min=('actual_normal_pct', 'min'), actual_normal_pct_max=('actual_normal_pct', 'max'), actual_attack_pct_mean=('actual_attack_pct', 'mean')).reset_index()
model_ratio_reference.to_csv(RESULTS_DIR / f'{MIXED_EXPERIMENT}_model_ratio_reference.csv', index=False)
mixed_overall.to_csv(RESULTS_DIR / f'{MIXED_EXPERIMENT}_all_runs.csv', index=False)
mixed_categories.to_csv(RESULTS_DIR / f'{MIXED_EXPERIMENT}_category_runs.csv', index=False)
mixed_subtypes.to_csv(RESULTS_DIR / f'{MIXED_EXPERIMENT}_subtype_runs.csv', index=False)
mixed_ratio_manifest.to_csv(RESULTS_DIR / f'{MIXED_EXPERIMENT}_test_ratio_manifest.csv', index=False)
mixed_per_source_summary.to_csv(RESULTS_DIR / f'{MIXED_EXPERIMENT}_per_source_summary.csv', index=False)
mixed_source_macro_by_seed.to_csv(RESULTS_DIR / f'{MIXED_EXPERIMENT}_source_macro_by_seed.csv', index=False)
mixed_source_macro_summary.to_csv(RESULTS_DIR / f'{MIXED_EXPERIMENT}_source_macro_summary.csv', index=False)
mixed_category_summary.to_csv(RESULTS_DIR / f'{MIXED_EXPERIMENT}_category_summary.csv', index=False)
ratio_summary.to_csv(RESULTS_DIR / f'{MIXED_EXPERIMENT}_test_ratio_summary.csv', index=False)
result_columns = ['model', 'seed', 'test_source', 'source_class_quota', 'n_train_used', 'n_test_raw', 'n_test_ratio_matched', 'n_test_evaluated', 'target_normal_ratio', 'target_attack_ratio', 'actual_normal_ratio', 'actual_attack_ratio', 'test_normal_pct', 'test_attack_pct', 'fit_seconds', 'prediction_seconds', 'n_test', 'tn', 'fp', 'fn', 'tp', 'accuracy', 'balanced_accuracy', 'precision', 'recall', 'specificity', 'f1', 'macro_f1', 'mcc', 'false_positive_rate', 'roc_auc']
source_order = ['CIC-IDS2017', 'CSE-CIC-IDS2018', 'DARPA', 'UNSW-NB15', 'POOLED_ALL_ROWS']
table_results = mixed_overall.copy()
table_results['__model'] = pd.Categorical(table_results['model'], categories=MODEL_KEYS, ordered=True)
table_results['__seed'] = pd.Categorical(table_results['seed'], categories=SEEDS, ordered=True)
table_results['__source'] = pd.Categorical(table_results['test_source'], categories=source_order, ordered=True)
table_results = table_results.sort_values(['__model', '__seed', '__source']).loc[:, result_columns]
expected_result_rows = len(MODEL_KEYS) * len(SEEDS) * len(source_order)
if len(table_results) != expected_result_rows:
    raise AssertionError(f'Expected {expected_result_rows} CIC17 result rows, got {len(table_results)}.')
table_results.to_csv(RESULTS_DIR / 'all_runs_CIC17.csv', index=False)
